In [1]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
from src.utils import paths

datasets = [
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-bs-v1/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-bs-v2/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-hr-v1/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-hr-v2/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/dgt-en-hr-v2021/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-2-bos/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-2-hrv/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-edu-10BT/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-hq-20BT/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/macocu-bs-en-sent/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/macocu-en-hr-sent/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/riznica/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/settimes-bs-en-v2/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/settimes-en-hr-v2/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/wikipedia-bs-20231101/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/wikipedia-en-20231101/tokenized.npy"),
    Path("/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/wikipedia-hr-20231101/tokenized.npy"),
]

for dataset in datasets:
    arr = np.load(dataset, mmap_mode="r")
    print(f"{dataset}: length = {len(arr)}")
weights = [1 for _ in datasets]
eos_token = 2
# chunk_size = 500_000_000
# ### precompute eos_token indices and save them
# for dataset in datasets:
#     ds = np.load(dataset, mmap_mode="r")
#     chunk = 0
#     eos_locs = []
#     while chunk < ds.size:
#         chunk_indices = np.where(ds[chunk:chunk+chunk_size] == eos_token)[0] + chunk
#         eos_locs.extend(chunk_indices.tolist())
#         chunk += chunk_size
#         np.save(dataset.parent / "eos_idx.npy", eos_locs)
#     print(len(eos_locs))

/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-bs-v1/tokenized.npy: length = 837016324
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-bs-v2/tokenized.npy: length = 1625512721
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-hr-v1/tokenized.npy: length = 2556134845
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/classla-web-hr-v2/tokenized.npy: length = 4821799605
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/dgt-en-hr-v2021/tokenized.npy: length = 177798354
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-2-bos/tokenized.npy: length = 13227509277
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-2-hrv/tokenized.npy: length = 8985441186
/media/mcvjetko/marko/transformer/data/processed/1B-model-vocab-48000/fineweb-edu-10BT/tokenized.npy: length = 10288617173
/media/mcvjetko/marko/trans

In [ ]:

class TokenMixtureDataset(IterableDataset):
    def __init__(
        self,
        datasets,
        weights,
        seq_len,
        eos_aware_sampling=False,
        eos_datasets=None,
        eos_token=None,
    ):
        self.ds = [np.load(p, mmap_mode="r") for p in datasets]
        self.weights = np.asarray(weights, dtype=np.float64)
        self.weights /= self.weights.sum()
        self.seq_len = seq_len
        self.eos_indices = None

        if eos_aware_sampling:
            assert eos_token is 
            is_eos = x == self.eos_token_idnot None

            paths = eos_datasets or [
                Path(p).with_suffix(".eos.npy") for p in datasets
            ]

            if eos_datasets is None:
                for src, dst in zip(datasets, paths):
                    if not dst.exists():
                        print(f"Preparing EOS indices for {src} -> {dst} ...")
                        self.prepare_eos_indices(src, dst, eos_token)
                   

            self.eos_indices = [
                np.load(p, mmap_mode="r") for p in paths
            ]

    @staticmethod
    def prepare_eos_indices(src, dst, eos_token, chunk_size=10_000_000):
        ds = np.load(src, mmap_mode="r")

        indices = [
            np.flatnonzero(ds[i:i+chunk_size] == eos_token) + i
            for i in range(0, len(ds), chunk_size)
        ]

        np.save(dst, np.concatenate(indices).astype(np.int64))

    def __iter__(self):
        rng = np.random.default_rng()
        n = self.seq_len + 1

        while True:
            idx = rng.choice(len(self.ds), p=self.weights)
            ds = self.ds[idx]

            if self.eos_indices is None:
                start = rng.integers(len(ds))
            else:
                eos = self.eos_indices[idx]
                start = int(eos[rng.integers(len(eos))]) + 1

            if start + n <= len(ds):
                yield ds[start:start+n]
            else:
                yield np.concatenate((ds[start:], ds[:start+n-len(ds)]))


Below, eos indice dataset, working

In [ ]:
from torch.utils.data import IterableDataset, get_worker_info

class TokenMixtureDatasetV2(IterableDataset):
    def __init__(
        self,
        datasets,
        weights,
        seq_len,
        seed,
    ):
        #self.ds = [np.load(p, mmap_mode="r") for p in datasets]
        self.weights = np.asarray(weights, dtype=np.float64)
        self.weights /= self.weights.sum()
        self.seq_len = seq_len
        self.eos_indices = None

    def __iter__(self):
        info = get_worker_info()
        worker = info.id if info else 0
        num_workers = info.num_workers if info else 1

        rank = torch.distributed.get_rank() if (
            torch.distributed.is_initialized()
        ) else 0
        world = torch.distributed.get_world_size() if (
            torch.distributed.is_initialized()
        ) else 1
         

    @staticmethod
    def prepare_eos_indices(src, dst, eos_token, chunk_size=10_000_000):
        ds = np.load(src, mmap_mode="r")

        indices = [
            np.flatnonzero(ds[i:i+chunk_size] == eos_token) + i
            for i in range(0, len(ds), chunk_size)
        ]

        np.save(dst, np.concatenate(indices).astype(np.int64))

    def __iter__(self):
        rng = np.random.default_rng()
        n = self.seq_len + 1

        while True:
            idx = rng.choice(len(self.ds), p=self.weights)
            ds = self.ds[idx]

            if self.eos_indices is None:
                start = rng.integers(len(ds))
            else:
                eos = self.eos_indices[idx]
                start = int(eos[rng.integers(len(eos))]) + 1

            if start + n <= len(ds):
                yield ds[start:start+n]
            else:
                yield np.concatenate((ds[start:], ds[:start+n-len(ds)]))


In [4]:
import math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler


class TokenDataset(Dataset):
    def __init__(self, path, seq_len=2048, seed=42):
        self.tokens = np.load(path, mmap_mode="r")
        self.seq_len = seq_len
        self.n = (len(self.tokens) - 1) // seq_len
        self.seed = seed

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        epoch, pos = divmod(int(idx), self.n)
        rng = np.random.default_rng(
            np.random.SeedSequence([self.seed, epoch])
        )

        # Stateless, without-replacement affine permutation
        if self.n == 1:
            a, b = 1, 0
        else:
            a = int(rng.integers(1, self.n))
            while math.gcd(a, self.n) != 1:
                a = int(rng.integers(1, self.n))
            b = int(rng.integers(self.n))

        start = ((a * pos + b) % self.n) * self.seq_len
        x = self.tokens[start:start + self.seq_len + 1]
        return torch.from_numpy(np.array(x, dtype=np.int64))


class WeightedMix(Dataset):
    def __init__(self, datasets, weights, total_tokens, seq_len=2048):
        self.datasets = datasets
        self.n = total_tokens // seq_len
        self.w = np.asarray(weights, dtype=np.float64)
        self.w /= self.w.sum()

        # Compact precomputed index, 5 bytes per sequence
        self.mapping = np.empty(
            self.n,
            dtype=[("src", "u1"), ("idx", "u4")]
        )
        counts = np.zeros(len(datasets), dtype=np.int64)

        for t in range(self.n):
            i = np.argmax(self.w * max(t, 1) - counts)
            self.mapping[t] = (i, counts[i])
            counts[i] += 1

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        src, pos = self.mapping[idx]
        return self.datasets[int(src)][int(pos)]


class DDPSampler(Sampler):
    def __init__(self, n, batch_size, rank, world_size):
        self.batch_size = batch_size
        self.rank = rank
        self.world_size = world_size
        self.steps = n // (batch_size * world_size)

    def __iter__(self):
        for step in range(self.steps):
            start = (
                step * self.world_size * self.batch_size
                + self.rank * self.batch_size
            )
            yield from range(start, start + self.batch_size)

    def __len__(self):
        return self.steps * self.batch_size


# Usage (after initializing torch.distributed)

seq_len = 2048
batch_size = 8

sources = [
    TokenDataset(path, seq_len, seed=42 + i)
    for i, path in enumerate(datasets)
]

mixed = WeightedMix(
    sources,
    weights=[1.0] * len(sources),
    total_tokens=100_000_000_000,
    seq_len=seq_len,
)

#rank = torch.distributed.get_rank()
#world_size = torch.distributed.get_world_size()
rank = 0
world_size = 1

loader = DataLoader(
    mixed,
    batch_size=batch_size,
    sampler=DDPSampler(len(mixed), batch_size, rank, world_size),
    num_workers=4,
    pin_memory=True,
)
import time

num_batches = 1000
start_time = time.time()

for i, tokens in enumerate(loader):
    tokens = tokens.cuda(non_blocking=True)
    x, y = tokens[:, :-1], tokens[:, 1:]

    if i + 1 >= num_batches:
        break

elapsed = time.time() - start_time
print(f"Processed {num_batches} batches in {elapsed:.2f} seconds "
      f"({num_batches / elapsed:.2f} batches/sec, "
      f"{(num_batches * batch_size) / elapsed:.2f} samples/sec)")

Processed 1000 batches in 2.59 seconds (386.01 batches/sec, 3088.10 samples/sec)


In [ ]:

import time
import numpy as np
from numba import njit

N = 1_000_000_000  # Change to 97_656_250 for your 200B-token run
W = np.arange(1, 18, dtype=np.float64)
W /= W.sum()


def python_mapping(w, n):
    counts = [0] * len(w)
    src = np.empty(n, dtype=np.uint8)
    idx = np.empty(n, dtype=np.uint32)

    for t in range(n):
        best = 0
        error = w[0] * max(t, 1) - counts[0]

        for i in range(1, len(w)):
            e = w[i] * max(t, 1) - counts[i]
            if e > error:
                best, error = i, e

        src[t] = best
        idx[t] = counts[best]
        counts[best] += 1

    return src, idx

jit_mapping = njit(python_mapping)

print(f"Samples: {N:,}")
print(f"Mapping memory: {N * 5 / 2**20:.1f} MiB\n")

# Compile separately (small input)
t0 = time.perf_counter()
jit_mapping(W, 10)
print(f"JIT compilation: {time.perf_counter() - t0:.3f} s\n")

# a = bench("Pure Python", python_mapping)
# b = bench("NumPy loop", numpy_mapping)
c = bench("Numba JIT", jit_mapping)

# assert np.array_equal(a[0], b[0])
# assert np.array_equal(a[1], b[1])
# assert np.array_equal(a[0], c[0])
# assert np.array_equal(a[1], c[1])

print("\nAll results identical.")

Samples: 1,000,000,000
Mapping memory: 4768.4 MiB

JIT compilation: 0.171 s

Numba JIT          22.350 s

All results identical.


In [2]:
divmod(5, 2)

(2, 1)

In [43]:
dataset_lens = np.array([12, 3, 10])
total_chunks = 25
weights = np.array([0.333, 0.333, 0.333], dtype=np.float32)

def build_sampling_plan(dataset_lens, total_chunks, weights, seed=42):

    rng = np.random.default_rng(seed)

    weights = weights / weights.sum()  # normalize weights
    chunks_per_dataset = np.round(weights * total_chunks).astype(int)

    chunks_per_dataset[np.argmax(dataset_lens)] += total_chunks - chunks_per_dataset.sum() 

    dataset_indices = np.array([])
    dataset_chunks = np.array([])
    for i in range(len(dataset_lens)):
        epochs, mod = divmod(chunks_per_dataset[i], dataset_lens[i])
        print(epochs, mod) 

        indices = np.array([], dtype=np.int32)
        while indices.size < chunks_per_dataset[i]:

            indices = np.concatenate((indices, rng.permutation(dataset_lens[i])))
        indices = indices[:chunks_per_dataset[i]]

        dataset_indices = np.concatenate((dataset_indices, np.full(indices.size, i)))
        dataset_chunks = np.concatenate((dataset_chunks, indices))

    # Shuffle indices and chunks jointly
    perm = rng.permutation(len(dataset_indices))
    dataset_indices = dataset_indices[perm]
    dataset_chunks = dataset_chunks[perm]

    return dataset_indices, dataset_chunks
ds_idx, chunk_idx = build_sampling_plan(dataset_lens, total_chunks, weights, seed=1)
print(ds_idx.shape)
print(chunk_idx.shape)
print(ds_idx)
print(chunk_idx)


class Dataset(torch.utils.data.Dataset):
    def __init__(self, ds_paths, ds_idx, chunk_idx):
        self.ds_idx = ds_idx
        self.chunk_idx = chunk_idx
        self.ds = [np.load(p, mmap_mode="r") for p in ds_paths]

    def __len__(self):
        return len(self.chunk_idx)

    def __getitem__(self, idx):
        return self.ds[self.ds_idx[idx]][self.chunk_idx[idx]]


0 9
2 2
0 8
(25,)
(25,)
[2. 1. 0. 1. 2. 2. 2. 0. 0. 1. 2. 0. 1. 0. 0. 1. 2. 2. 2. 0. 0. 0. 1. 1.
 1.]
[ 3.  0.  2.  1.  7.  2.  9.  4.  7.  0.  1.  8.  1.  9.  1.  0.  6.  8.
  0.  5.  0. 11.  2.  1.  2.]
